# Cante Style Embeddings - Cantaor Identification

Portfolio project in computational flamenco musicology. The goal is to learn an embedding space that captures each cantaor's **interpretive style**, not *what* palo they sing, but *how* they sing it, following the same logic as speaker embeddings in speech recognition.

This notebook documents the first step: an exploratory analysis of the two available datasets (TONAS and cante100) to decide, based on real data rather than assumptions, which evaluation protocol is viable given the sample size.

In [ ]:
import os
print(os.getcwd())

In [2]:
import pandas as pd
import re
import unicodedata

c100 = pd.read_csv('data/raw/cante100_catalogo - cante100_catalogo.csv.csv')
tonas = pd.read_csv('data/raw/tonas_catalogo - tonas_catalogo.csv.csv')

print(f"cante100: {c100.shape[0]} pistas, {c100.shape[1]} columnas")
print(f"TONAS: {tonas.shape[0]} pistas, {tonas.shape[1]} columnas")

cante100: 100 pistas, 11 columnas
TONAS: 72 pistas, 12 columnas


## Normalizing cantaor names

Cantaor names appear inconsistently across both datasets: with or without accents, with nicknames in quotes (`Manuel Soto "Sordera"`), with stray articles, or with parenthetical caveats (`M. Vargas (nombre de pila sin confirmar)`). To compare and combine both datasets by cantaor, we need a function that reduces any variant of a name to a single normalized key.

In [13]:
def norm(name):
    if pd.isna(name):
        return ""
    s = str(name)
    s = re.sub(r'\(.*?\)', '', s)          # quita paréntesis, ej. "(nombre de pila sin confirmar)"
    s = s.replace('"', '')                  # quita comillas
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')  # quita acentos
    s = s.lower().strip()
    s = re.sub(r'\bel\b|\bla\b|\bde\b|\bdel\b', '', s)  # quita articulos sueltos
    s = re.sub(r'[^a-z]', '', s)            # se queda solo con letras
    return s

# Comprobación rápida
for ejemplo in ['Manuel Soto "Sordera"', 'M. Vargas (nombre de pila sin confirmar)', 'Tía Anica "La Piriñaca"']:
    print(f"{ejemplo!r:55} -> {norm(ejemplo)}")

'Manuel Soto "Sordera"'                                 -> manuelsotosordera
'M. Vargas (nombre de pila sin confirmar)'              -> mvargas
'Tía Anica "La Piriñaca"'                               -> tiaanicapirinaca


## Deduplicating repeated audio in cante100

The cante100 catalog includes a `hash_audio_1mb` column, computed from the first megabyte of each audio file's content. Two pairs of tracks share an identical hash despite having different filenames, meaning the same recording was catalogued twice. These duplicates must be removed before counting recordings per cantaor, or the count would be inflated.

In [14]:
c100_dedup = c100.drop_duplicates(subset='hash_audio_1mb', keep='first').copy()
c100_dedup['norm_name'] = c100_dedup['cantaor_normalizado'].apply(norm)

print(f"cante100: {len(c100)} tracks -> {len(c100_dedup)} after removing duplicate audio")

cante100: 100 tracks -> 98 after removing duplicate audio


## Filtering TONAS by mapping confidence

The TONAS catalog includes a `confianza_mapeo` column (high/medium/low), indicating how confident the cantaor identity mapping is. Tracks with low confidence correspond to ambiguous initials (e.g. "M. Simón") with no recoverable full name in the dataset. These are excluded from the reliable count, since a doubtful identity label would contaminate a contrastive loss downstream.

In [15]:
tonas['norm_name'] = tonas['cantaor_propuesto'].apply(norm)
tonas_fiable = tonas[tonas['confianza_mapeo'].isin(['alta', 'media'])].copy()

print(f"TONAS: {len(tonas)} tracks -> {len(tonas_fiable)} with medium/high confidence "
      f"(excludes {len(tonas) - len(tonas_fiable)} low-confidence tracks)")

TONAS: 72 tracks -> 59 with medium/high confidence (excludes 13 low-confidence tracks)


## Counting recordings per cantaor

We group each (deduplicated, filtered) dataset by normalized name and count how many recordings each cantaor has, then combine both datasets by summing counts for matching names.

In [16]:
c100_counts = c100_dedup.groupby('norm_name').size().rename('n_c100')
c100_names = c100_dedup.groupby('norm_name')['cantaor_normalizado'].first().rename('nombre_c100')

tonas_counts = tonas_fiable.groupby('norm_name').size().rename('n_tonas')
tonas_names = tonas_fiable.groupby('norm_name')['cantaor_propuesto'].first().rename('nombre_tonas')

combined = pd.concat([c100_counts, c100_names, tonas_counts, tonas_names], axis=1)
combined['n_c100'] = combined['n_c100'].fillna(0).astype(int)
combined['n_tonas'] = combined['n_tonas'].fillna(0).astype(int)
combined['total'] = combined['n_c100'] + combined['n_tonas']
combined['nombre'] = combined['nombre_c100'].fillna(combined['nombre_tonas'])
combined = combined.sort_values('total', ascending=False)

combined[['nombre', 'n_c100', 'n_tonas', 'total']].head(15)

,nombre,n_c100,n_tonas,total
norm_name,,,,
antoniomairena,Antonio Mairena,8,6,14
manuelsotosordera,"Manuel Soto ""Sordera""",7,0,7
juantalega,Juan Talega,2,4,6
fernandoterremoto,Fernando Terremoto,6,0,6
camaron,Camarón,5,0,5
diegoclavel,Diego Clavel,3,2,5
manuelagujetas,Manuel Agujetas,4,1,5
antonionunezchocolate,"Antonio Nunez ""Chocolate""",0,4,4
fosforito,Fosforito,4,0,4


## Merging the Rafael Romero cross-dataset match

The TONAS notes explicitly flag one identity that the automatic normalization didn't catch: "Rafael Romero" (cante100) and 'Rafael Romero "El Gallina"' (TONAS) refer to the same person (the nickname in quotes wasn't stripped by `norm()`). This kind of cross-dataset identity resolution can't be fully automated without risking false merges between different people, so it's corrected manually here, based on the explicit note in the source catalog.

In [17]:
rr = combined.loc[['rafaelromero', 'rafaelromerogallina']]
merged_row = pd.DataFrame({
    'n_c100': [rr['n_c100'].sum()],
    'nombre_c100': ['Rafael Romero'],
    'n_tonas': [rr['n_tonas'].sum()],
    'nombre_tonas': ['Rafael Romero "El Gallina"'],
    'total': [rr['total'].sum()],
    'nombre': ['Rafael Romero']
}, index=['rafaelromero_merged'])

combined = combined.drop(['rafaelromero', 'rafaelromerogallina'])
combined = pd.concat([combined, merged_row]).sort_values('total', ascending=False)

print(f"Unique cantaores (after merging the Rafael Romero match): {len(combined)}")
print()
print(f"With >=3 recordings: {(combined['total'] >= 3).sum()}")
print(f"With 2 recordings:   {(combined['total'] == 2).sum()}")
print(f"With 1 recording:    {(combined['total'] == 1).sum()}")

combined.to_csv('data/processed/conteo_cantaores_combinado.csv')

Unique cantaores (after merging the Rafael Romero match): 79

With >=3 recordings: 16
With 2 recordings:   15
With 1 recording:    48


## Conclusion: protocol decision

Only 16 cantaores have 3+ recordings and 15 have exactly 2, out of 79 unique cantaores total. This sample size is too small to support a genuine open-set evaluation protocol (verification against cantaores never seen in training) with statistical confidence: reserving even 4-6 cantaores for open-set evaluation would leave the headline result resting on too few cases to be defensible.

Given this, the primary evaluation protocol for this project is **closed-set pairwise verification**: the same ~31 cantaores with 2+ recordings appear in both training and test, split at the recording level (never the same recording in both), with distinct recordings held out for testing per cantaor. Equal Error Rate (EER) and ROC/AUC will be the primary metrics, following standard practice in speaker verification.

An open-set exploration with the handful of cantaores available for it is planned as a secondary, exploratory analysis, reported explicitly as preliminary given the small *n*, not as a headline result.